# FluxFit Demo

This notebook walks through a complete fit, from raw data to circuit parameters with
uncertainties, on **synthetic data whose answer is known**, so you can check at the end
that the numbers came out right.

It takes about five minutes, most of which is one 50 s global search.

**Before you start.** Put `fluxfit.py` next to this notebook and make sure you have:

```
pip install numpy scipy matplotlib scikit-image scqubits ipywidgets ipympl
```

`ipympl` is the one people forget. Without it the plots are static pictures and none of
the clicking works. The first cell checks for you.

In [1]:
%matplotlib widget
import numpy as np
import matplotlib
from fluxfit import Scan, ModelSpec, FluxoniumFitter, synthetic_scan

assert "ipympl" in matplotlib.get_backend().lower(), (
    "the backend is " + matplotlib.get_backend() + ": run `%matplotlib widget` "
    "in this cell (and `pip install ipympl` if that fails)")
print("ready, backend:", matplotlib.get_backend())

ready, backend: module://ipympl.backend_nbagg


---
## Step 1 - get your data into a `Scan`

A `Scan` is just three numpy arrays and a name:

```python
Scan("wide", x=coil_values, f=frequencies, z=signal)
```

- `x` - whatever you swept: coil voltage, current, or flux itself. One number per column.
- `f` - the drive frequency axis. One number per row.
- `z` - the measured signal, real or complex, shaped `(len(f), len(x))` or its transpose.
  The orientation is worked out for you, and both axes are sorted.

Two things that catch people out:

- **Units.** `f` is assumed to be in GHz. If yours is in MHz or Hz, say so with
  `f_unit="MHz"`. Getting this wrong is silent otherwise, and your lines end up at
  180 GHz instead of 180 MHz.
- **Already calibrated?** If `x` is in flux quanta rather than volts, pass
  `x_unit="Phi0"` and the calibration is fixed to the identity.

Normalising `z` is pointless, so don't bother: everything downstream uses a robust
z-score, which is scale-invariant.

For a real experiment this cell would be your file loader. Here we fabricate two scans
from a known fluxonium so that we can grade the fit at the end. The generated data has
12 MHz lines, a 3 MHz-per-column random wander of each line, 15 % of the lines
attenuated, and noise at 1/8 of a strong line's height.

In [2]:
TRUE = dict(EJ=4.1, EC=1.0, EL=0.4, E_osc=7.0, g=0.1, x0=0.1, slope=0.7)
# x0 and slope mean: Phi/Phi_0 = 0.7 * (x - 0.1 V), i.e. a period of 1.4286 V

spec = ModelSpec(with_resonator=False, n_fluxonium=6, from_levels=(0, 1), matelem_min=0.03)

wide = synthetic_scan("wide", (4.0, 6.5), 321, (-1.15, 1.05), 241,
                      TRUE, spec, snr=8.0, faint=0.15, seed=3)
zoom = synthetic_scan("zoom", (0.15, 2.3), 221, (0.62, 0.82), 121,
                      TRUE, spec, snr=8.0, faint=0.0, seed=4)
scans = [wide, zoom]

for s in scans:
    print(f"{s.name}: {s.f.size} frequencies from {s.f[0]:.3f} to {s.f[-1]:.3f} GHz, "
          f"{s.x.size} coil values from {s.x[0]:.3f} to {s.x[-1]:.3f} {s.x_unit}")

wide: 321 frequencies from 4.000 to 6.500 GHz, 241 coil values from -1.150 to 1.050 V
zoom: 221 frequencies from 0.150 to 2.300 GHz, 121 coil values from 0.620 to 0.820 V


Why two scans? The wide one shows the level structure over more than one flux
period, which is what pins `E_J` and `E_C`. The low-frequency zoom catches the minimum of
the 0-1 transition at half flux, which is the single most informative point for `E_L` and
the one that tells you exactly where half flux is. Fitting them together is the point of
the tool.

**Your own data:** replace the two `synthetic_scan` calls with your loader, for example

```python
wide = Scan("wide", x=voltages, f=freqs_GHz, z=my_complex_data)
zoom = Scan("zoom", x=v2, f=f2_MHz, z=z2, f_unit="MHz")
```

---
## Step 2 - open the tool

The next cell opens the interface. Leave it running and keep scrolling: the following
cells tell you which buttons to press, and the interface stays live above them.

What you are looking at, top to bottom:

- **Two plots**, one per scan, each with the matplotlib toolbar underneath (zoom, pan,
  home). While a toolbar tool is active, clicks go to the toolbar and not to the tool.
- **A row of mode buttons** (`toggle`, `label`, `select`, `draw`, `erase`, `trim`,
  `calibrate`) that decide what a left click on a plot does, a **transition dropdown**
  used by the `label` mode, and a status line describing the current mode.
- **Six tabs**: View, Detect, Calibration, Model, Fit, Lines.
- **A log** at the bottom. Read it. Every action writes there, including the warnings
  that matter.

In [3]:
app = FluxoniumFitter(scans, with_resonator=False)

app.set_spec(spec)                                 # 6 levels, transitions from |0> and |1>
app.params.update(EJ=5.0, EC=1.3, EL=0.8)          # a deliberately wrong starting point
app.bounds.update(EJ=(2.0, 8.0), EC=(0.5, 2.0), EL=(0.1, 1.5))

app.show()

---
## Step 3 - preprocessing (View tab)

Everything downstream works on a **robust z-score map**, not on your raw data: each
column has its slow baseline removed and is divided by a robust estimate of the noise. A
detection threshold is then in units of the noise standard deviation, and the same number
works on scans taken at different drive powers.

`app.show()` already ran this once with the defaults. Open the **View** tab and look:

1. **signal** - which part of a complex `z` to use. `abs` is the usual choice; `dphase`
   (the phase gradient) sometimes shows faint lines better.
2. **peaks** - are your lines bright or dark in the processed map? `auto` guesses from the
   percentiles. If the lines look inverted, set `positive` or `negative` explicitly.
3. **noise est.** - keep `global`. The `column` option divides every flux column by its
   own noise, which sounds sensible but destroys a line that fills the frequency window of
   a narrow zoom.
4. **remove flux-independent bg** - only tick this for scans spanning a good fraction of a
   period. In a narrow zoom it deletes a nearly flat line along with the background.
5. **detrend f** - window of the baseline filter, as a fraction of the frequency span. It
   turns itself off below 9 points and says so in the log.

Press **Apply / preprocess** after any change. The log prints the maximum z-score per
scan: a strong line should be tens of sigma, a faint one a few.

---
## Step 4 - calibration (Calibration tab)

The tool needs to know how your coil value maps to flux:

```
Phi/Phi_0 = slope * (x - dx_scan - x0)
```

`dx_scan` is a per-scan offset for drift between recordings. The first scan is the
**reference** and has `dx = 0`; every other scan gets its own.

Press **Auto-calibrate**. It finds the flux period from the autocorrelation of the image
and the mirror axes from the image's symmetry, with no Hamiltonian involved. On this
synthetic data it returns a period of 1.430 V against a true 1.4286 V, an error of 0.1 %.

**Then check it, because it gets two things wrong regularly:**

- **`Phi = 0` and `Phi = 1/2` swapped.** The tool guesses from where the spectral weight
  sits and is often wrong. The giveaway is that the 0-1 line should be at its *minimum* at
  half flux. If the overlay has it backwards, press **flip 0 <-> 1/2**.
- **The period off by a factor of two.** Symmetry axes recur every half period, so the
  autocorrelation can lock onto `P/2`. If the model lines are twice too dense or twice too
  sparse in flux, press **period x 2** or **period / 2**.

**Setting it by hand** is often better than the heuristic. Choose what a click means under
*click sets*, switch to `calibrate` mode, and click on a plot. The most useful
combination, when you have a low-frequency zoom:

1. take the period from the wide scan (auto-calibration, or two clicks with
   *Phi = 1/2 (set period)*);
2. choose **Phi = 1/2 (keep period)**;
3. click the minimum of the 0-1 line in the zoom.

That moves only the offset, keeping the period you trust. And since a click calibrates the
scan it was made in, clicking on the drifted zoom adjusts *its* offset without displacing
the wide scan.

If a click produces an offset larger than 0.25 Phi_0, the log tells you: that is not
drift, it means `Phi = 0` and `Phi = 1/2` are swapped in the reference scan.

---
## Step 5 - a starting point without labels (Fit tab)

Now the part that saves the most time. Before any line is detected or labelled, press
**Coarse search**.

It works by tracing every transition the current parameters predict through the image and
summing the measured signal underneath it, weighted by the charge matrix element
`|<i|n|j>|`, with a penalty per sampled point so that a line lying on background is a net
loss. A differential-evolution search maximises that over `E_J`, `E_C`, `E_L`. No lines
and no labels are needed, so even a line too faint to be detected contributes.

Takes about 50 s here. **Judge the result on the overlay, not on the score**: the dashed
model curves should sit on the bright features across the whole flux range. Two patterns
worth recognising:

- matches near half flux, drifts at the edges -> the flux period is wrong, go back to
  Step 4;
- lines in the gaps -> check the polarity in the View tab, then your bounds.

Before pressing it, set bounds you actually believe in the **Model** tab. The defaults
(`E_J` from 0.5 to 20 GHz) are wide enough to be slow and to admit aliases. And drag the
`E_J`, `E_C`, `E_L` sliders by hand for half a minute first: the overlay updates live, and
it tells you immediately whether you are in the right regime.

The coarse search is the least reliable part of the tool. It needs several distinct
branches inside your frequency window; with only three or four it can converge somewhere
wrong. If it does, set the parameters by hand with the sliders and carry on to Step 6 -
everything after this point works from any reasonable starting point.

---
## Step 6 - detect the lines (Detect tab)

Press **Detect (all)** with *threshold* around 2.5 sigma and *min length* 12.

What happens: in each column the peaks are located and refined to sub-pixel accuracy, then
tracks are grown column by column, predicting the next frequency from the slope of the
last few points and matching predictions to peaks. You get whole lines, not points.

You should see about 11 traces drawn in white over the wide scan. Then:

- **A faint branch was missed?** Zoom onto it with the matplotlib toolbar, lower the
  threshold to 1.5, press **Detect in view**. It only looks inside the current axes limits,
  so you do not flood the rest of the image with noise.
- **Still nothing?** Tick **ridge filter** and raise *min length* to 30. The filter follows
  faint lines much further - on a 2.7 sigma line it returns one continuous trace where
  plain peak finding returns three fragments - at the cost of short spurious segments,
  which the length cut removes.
- **Hundreds of junk segments?** Threshold too low, or ridge filter on with a small
  *min length*.

**Clear auto lines** removes detected traces but keeps hand-drawn ones, so you can
re-detect with different settings without losing your work.

---
## Step 7 - curate and label by clicking

The mode buttons above the tabs decide what a left click does. A click acts on the nearest
line within 12 pixels; if nothing is that close, the status line says so.

| mode | a left click does |
|---|---|
| `toggle` | include the line in the fit, or exclude it. Excluded lines stay visible, thin and dashed |
| `label` | give the line the transition chosen in the dropdown next to the mode buttons |
| `select` | select a line so its numbers appear in the Lines tab |
| `draw` | add a point. After three or four, press **Finish line** in the Detect tab |
| `erase` | delete the line |
| `trim` | cut the line at that flux, on the side nearer your click |
| `calibrate` | set the flux reference chosen in the Calibration tab |

**Drawing a line by hand** is how you deal with anything the detector missed, typically the
0-1 line in a narrow zoom. Switch to `draw`, click three or four points roughly along the
line, press **Finish line**: the clicks are interpolated and then each column is snapped to
the local maximum within the *snap* window, so a rough gesture becomes a properly extracted
trace.

**Labelling by hand** is one click: pick the transition in the dropdown, switch to `label`,
click the line. A hand-set label is marked `*` in the Lines tab and the automatic
assignment will not overwrite it - which matters for the zoom's 0-1 line, the one you are
most sure about and the one the automatic step is least able to place. **unlock hand
labels** releases them.

Everything is also available from the **Lines** tab, which is easier when lines overlap:
one row per line with a checkbox, a `sel` button, a delete button, the label, `conf` and
the residual.

---
## Step 8 - assign and refine (Fit tab)

Press **Assign + refine loop**.

Assignment gives each trace the transition whose predicted frequencies have the smallest
median distance to it, if that distance is under the tolerance. Refinement is then least
squares on the labelled points with a robust loss.

The loop does both three times, from a 150 MHz tolerance down to the slider value. A
single pass is usually not enough: the first assignment only labels what your starting
model already explains, and on a real dataset one pass at 50 MHz labelled 3 traces of 15
while the loop labelled all 15.

Then read the **Lines** tab:

- **res** - the median distance to the assigned transition, in MHz.
- **conf** - the ratio of the second-best transition's distance to the best. `conf = 5` is
  safe; `conf` near 1 means two transitions pass equally close and the label is a coin
  flip. Fix those by hand (Step 7) and press refine again.

And read the log. Two warnings matter:

- *"N free parameters against M distinct transitions"* - one transition constrains one
  combination of `E_J`, `E_C` and `E_L`, not the three separately. Label more branches.
- *"Jacobian condition number ..."* above 1e4 - the fit is underdetermined and the
  individual uncertainties are meaningless in isolation.

**Do not tick `x0` and `slope`** until the fit is converged and most lines are labelled.
They start fixed for a reason: releasing them together with the energies from a good
starting point has been measured to diverge to an rms residual of 1052 MHz.

In [4]:
# how did we do? (only possible because this is synthetic data)
print(f"{'':6s} {'fitted':>12s} {'sigma':>10s} {'true':>9s} {'deviation':>11s}")
for k in ("EJ", "EC", "EL"):
    s = app.result["sigma"].get(k, float("nan"))
    print(f"{k:6s} {app.params[k]:12.5f} {s:10.5f} {TRUE[k]:9.4f} "
          f"{100*(app.params[k]-TRUE[k])/TRUE[k]:+10.2f} %")
print(f"\nrms residual {app.result['rms_GHz']*1e3:.1f} MHz on {app.result['n_points']} "
      f"points, condition number {app.result['cond']:.0f}")
print(f"flux period {1/app.params['slope']:.4f} (true {1/TRUE['slope']:.4f})")
print(f"lines labelled: {sum(t.label is not None for t in app.traces)} of {len(app.traces)}")

             fitted      sigma      true   deviation
EJ          4.10233    0.00474    4.1000      +0.06 %
EC          0.99354    0.00089    1.0000      -0.65 %
EL          0.40360    0.00041    0.4000      +0.90 %

rms residual 17.0 MHz on 181 points, condition number 16
flux period 1.4300 (true 1.4286)
lines labelled: 11 of 11


Expected, give or take the random seed of your run:

| quantity | fitted | sigma | true | deviation |
|---|---|---|---|---|
| E_J | 4.0961 GHz | 0.0049 | 4.1 | -0.10 % |
| E_C | 0.9965 GHz | 0.0012 | 1.0 | -0.35 % |
| E_L | 0.4024 GHz | 0.0006 | 0.4 | +0.61 % |

**Notice that the deviations are several times the quoted `sigma`.** That is not a bug and
it is the single most important thing to take away from this notebook. The uncertainty
comes from the Jacobian and describes how well the *model* fits the *extracted points*. It
knows nothing about the bias of the extraction itself - the line wander, the median
smoothing, the faint lines pulled by noise - which is coherent and does not average away.

Treat about 1 % as the real error floor, or measure it: refit with different detection
settings and take the spread of the answers.

---
## Step 9 - save your session

```python
app.save("device_A.json")
```

writes the parameters, bounds, free/fixed flags, the model specification and every trace
with its label and inclusion flag. The raw data is **not** saved, so restoring is

```python
app = FluxoniumFitter(scans, with_resonator=False)   # the same scans, same names
app.load("device_A.json")
app.show()
```

In [5]:
app.save("demo_session.json")

---
## Doing it without the interface

Every step is a plain method, so a fit can be scripted or looped over many devices. This
is the whole notebook in one cell:

```python
app = FluxoniumFitter(scans, with_resonator=False)
app.params.update(EJ=5.0, EC=1.3, EL=0.8)
app.bounds.update(EJ=(2.0, 8.0), EC=(0.5, 2.0), EL=(0.1, 1.5))
app.preprocess()
app.auto_calibrate()     # also narrows the bounds on x0, slope and the offsets
app.run_coarse()
app.detect()
app.polish()             # assign and refine, three times, tightening
print(app.params, app.result["sigma"], app.result["rms_GHz"])
```

`app.guided_run()` is exactly that sequence in one call, and the **Guided run** button
presses it for you.

The lower-level functions take arrays and dictionaries and carry no interface state:
`significance_map`, `detect_traces`, `snap_to_ridge`, `auto_calibration`, `image_score`,
`coarse_search`, `assign_traces`, `refine_fit`, `residual_table`.

---
## Fitting with a resonator

Tick **couple a resonator** in the Model tab, or pass `with_resonator=True`, and the model
becomes a fluxonium coupled to one harmonic mode,

```
H = H_fluxonium + E_osc a^dag a + g n (a + a^dag)
```

which is the same Hamiltonian as the scqubits `HilbertSpace` construction with
`op1=n_operator`, `op2=creation_operator`, `add_hc=True`. Two new parameters, `E_osc` and
`g`, appear in the table, and lines are labelled by the bare product states they overlap
most: `|0,0>-|2,0>` for a qubit transition, `|0,0>-|0,1>` for the resonator branch.

One thing to know: **`g` is the parameter most exposed to extraction bias**, because it is
set by the avoided crossing, which is exactly where the tracker tends to follow the
diabatic line straight across the gap. On synthetic data with `g = 0.120 GHz` the
automatic chain returned `0.102 +- 0.005 GHz`, a 15 % error, while `E_J`, `E_C`, `E_L` and
`E_osc` all came back within 0.05 %. If `g` matters to you, use `trim` mode to cut the
traces on either side of the crossing, or draw the two branches through it by hand.

In [6]:
res_true = dict(EJ=4.1, EC=1.0, EL=0.4, E_osc=5.6, g=0.12, x0=0.1, slope=0.7)
res_spec = ModelSpec(with_resonator=True, n_fluxonium=5, n_resonator=3,
                     from_levels=(0, 1), matelem_min=0.03)
res_scan = synthetic_scan("res", (4.0, 6.2), 301, (-1.15, 1.05), 181,
                          res_true, res_spec, snr=8.0, seed=5)

app2 = FluxoniumFitter([res_scan], with_resonator=True)
app2.set_spec(res_spec)
app2.params.update(EJ=4.5, EC=1.15, EL=0.55, E_osc=5.4, g=0.2, x0=0.1, slope=0.7)
app2.bounds.update(EJ=(3.0, 6.0), EC=(0.6, 1.6), EL=(0.2, 0.9),
                   E_osc=(5.0, 6.2), g=(0.0, 0.4))
app2.show()

---
## If something goes wrong

| symptom | cause | what to do |
|---|---|---|
| clicking does nothing | `%matplotlib widget` not active, or a toolbar tool is selected | rerun the first cell; deselect zoom/pan |
| a line you can see is absent from the map | frequency unit wrong, or the wrong `noise est.` | check `f_unit`; set `noise est.` to `global` |
| no lines found in a narrow zoom | flux-independent background removal deleted a flat line | untick it and reprocess |
| coarse search returns nonsense | lines are minima, not maxima | set `peaks` explicitly in the View tab |
| model matches at half flux, drifts at the edges | flux period wrong | *period x2* or */2*, or recalibrate by clicking |
| model lines twice too dense in flux | period aliased to `P/2` | **period x 2** |
| everything labelled the same | the model is wrong, not the assignment | go back to the coarse search or the sliders |
| `conf` near 1 on many lines | two branches genuinely overlap there | trim the traces to where the branches separate |
| condition number above 1e4 | too few distinct transitions labelled | label more branches, or fix some parameters |
| a parameter runs to its bound | it is unconstrained by the selected lines | untick it |
| the fit is slow | too many distinct flux values | lower *points / line* in the Fit tab |